# 03 - Evaluate and predict

Use this notebook after training. It validates the trained single-class fracture model and writes prediction images under `runs/fracture/predict`.


In [ ]:
RUN_ENV = "local"  # "colab" or "kaggle"
PROJECT_NAME = "yolov8-fracture-detection"
IMAGE_SIZE = 640
CONFIDENCE = 0.25


In [ ]:
from pathlib import Path

if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
elif RUN_ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working") / PROJECT_NAME
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DATA_ROOT = PROJECT_ROOT / "data"
RUNS_ROOT = PROJECT_ROOT / "runs"
WEIGHTS_PATH = PROJECT_ROOT / "weights" / "fracture_yolov8n_best.pt"

# Locate the YAML from the Roboflow download (01_dataset_setup.ipynb)
fracture_dir = DATA_ROOT / "fracture"
yaml_files = list(fracture_dir.glob("*.yaml"))
if not yaml_files:
    raise FileNotFoundError(f"No YAML found in {fracture_dir} — run 01_dataset_setup.ipynb first.")
DATA_YAML = yaml_files[0]

print(f"Dataset YAML : {DATA_YAML}")
print(f"Weights      : {WEIGHTS_PATH}")

In [ ]:
if RUN_ENV in {"colab", "kaggle"}:
    %pip install -U ultralytics


In [ ]:
from ultralytics import YOLO

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Missing dataset YAML: {DATA_YAML}")
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Missing trained weights: {WEIGHTS_PATH}\nRun 02_train_yolov8_fracture.ipynb first.")

model = YOLO(str(WEIGHTS_PATH))
metrics = model.val(
    data=str(DATA_YAML),
    imgsz=IMAGE_SIZE,
    project=str(RUNS_ROOT / "fracture"),
    name="val",
    exist_ok=True,
)
metrics

In [ ]:
# Predict on the test split. Change SOURCE_IMAGES to any folder of X-rays you want to inspect.
SOURCE_IMAGES = DATA_YAML.parent / "test" / "images"

prediction_results = model.predict(
    source=str(SOURCE_IMAGES),
    imgsz=IMAGE_SIZE,
    conf=CONFIDENCE,
    save=True,
    project=str(RUNS_ROOT / "fracture"),
    name="predict",
    exist_ok=True,
)
print(f"Prediction images saved under: {RUNS_ROOT / 'fracture' / 'predict'}")

In [ ]:
# Display a few saved predictions in the notebook.
from IPython.display import Image, display

predict_dir = RUNS_ROOT / "fracture" / "predict"
for image_path in sorted(predict_dir.glob("*.jpg"))[:6]:
    display(Image(filename=str(image_path)))
